# Explainability Evaluation — KRAG

This notebook reproduces the explainability and faithfulness workflow used for the KRAG evaluation.

| Part | Metric | What it measures |
|------|--------|------------------|
| IV.E.1 | FNS (Faithfulness Necessity Score) | Causal reliance on knowledge-graph evidence |
| IV.E.2 | RAGAS Faithfulness | Explanation fidelity to graph context |

FNS uses perturbation analysis and does not require an LLM. RAGAS faithfulness uses an LLM judge.

---

### Before you start
| # | Do this |
|---|--------|
| 1 | Set the runtime to **GPU** |
| 2 | Set `KRAG_REPO_URL` and `GOOGLE_CLOUD_PROJECT` in the setup cells below |
| 3 | Have the trained encoder checkpoint ready to upload |

Run cells top-to-bottom. One runtime restart is required after the install cell.

## Install & clone

In [ ]:
import os
import subprocess
import sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args, "-q"])

_pip("torch-geometric", "google-genai")

REPO_URL = os.environ.get("KRAG_REPO_URL", "https://github.com/<owner>/<repo>.git")
REPO_DIR = "KRAG"
!git clone {REPO_URL} {REPO_DIR}

_pip("-r", f"{REPO_DIR}/requirements.txt")
print("Done. Restart the runtime, then continue.")

### Restart the runtime
Go to **Runtime > Restart runtime**, then continue from the next cell.

## Authenticate to Google Cloud

In [ ]:
from google.colab import auth
auth.authenticate_user()

import os
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

from google.cloud import storage
storage.Client(project=PROJECT_ID)
print(f"Authenticated project: {PROJECT_ID}")

## Upload trained encoder

In [ ]:
from google.colab.files import upload
print("Upload the trained KRAG encoder checkpoint")
uploaded = upload()
ENCODER_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {ENCODER_PATH}")

## Imports

In [ ]:
import sys, torch, json, numpy as np
from scipy import stats as sp_stats
sys.path.insert(0, "KRAG/src")

from krag.system               import ARAGSystem, ARAGSystemConfig
from krag.training.gnn_trainer  import prepare_emotion_ground_truth
from krag.data.adapters         import get_adapter
from krag.evaluation.synthetic_testset import SyntheticTestSetGenerator
from krag.evaluation.metrics    import (
    compute_affective_precision_at_k,
    compute_affective_displacement_error,
    compute_semantic_recall_at_k,
    compute_faithfulness_necessity_score,
)
from krag.evaluation.ragas_evaluator import (
    RAGASFaithfulnessEvaluator,
    build_full_subgraph_context,
)
from krag.retrieval.bm25_retriever  import create_bm25_retriever_from_content_items
from krag.retrieval.krag_retriever  import QueryContext
from krag.core.emotion_detection    import EmotionProfile

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Imports OK | device = {DEVICE}")

## Initialize system

In [ ]:
config = ARAGSystemConfig()
config.vertex_ai_project = PROJECT_ID
system = ARAGSystem(config)
system.initialize()
stats  = system.load_and_index_data()
print(f"Content items : {stats['content_items']}")
print(f"KG nodes      : {stats['knowledge_graph_nodes']}")
print(f"KG edges      : {stats['knowledge_graph_edges']}")

## Load encoder and re-index at k=2

In [ ]:
system.krag_encoder.load_state_dict(
    torch.load(ENCODER_PATH, map_location=DEVICE)
)
system.krag_encoder.to(DEVICE)
system.krag_encoder.eval()

system.response_generator.generate_response = lambda *a, **kw: ""

content_ids = [str(item.id) for item in system.content_items]
system.subgraph_retriever.subgraph_embeddings = {}
system.subgraph_retriever.index_subgraphs(content_ids, hops=2)
print("Encoder loaded, subgraphs re-indexed at k=2.")

## Load evaluation data

In [ ]:
adapter = get_adapter()
movies_df = adapter.load_movies(vector_ready=True)
aff_sigs  = prepare_emotion_ground_truth(movies_df)
print(f"Affective signatures : {len(aff_sigs)}")

_result = system.vector_store.semantic_collection.get(include=["embeddings"])
_emb_map = dict(zip(_result["ids"], _result["embeddings"]))
content_embeddings = np.array([_emb_map[cid] for cid in content_ids])
print(f"Content embeddings   : {content_embeddings.shape}")

## Generate test sets

800 agreement + 800 dissonance queries, identical to the comparative analysis (seed=42).

In [ ]:
gen = SyntheticTestSetGenerator(
    content_embedder=system.content_embedder,
    semantic_threshold=0.5,
    affective_threshold=0.6,
    seed=42,
)

print("[1/2] Agreement queries ...")
agreement = gen.generate_test_set(
    content_items=system.content_items,
    content_embeddings=content_embeddings,
    movie_affective_signatures=aff_sigs,
    num_queries=800,
    min_relevant=3,
)
print(f"      {len(agreement)} queries")

print("[2/2] Dissonance queries ...")
dissonance = gen.generate_dissonance_queries(
    content_items=system.content_items,
    content_embeddings=content_embeddings,
    movie_affective_signatures=aff_sigs,
    num_queries=800,
)
print(f"      {len(dissonance)} queries")

## Create retrievers

In [ ]:
bm25_retriever = create_bm25_retriever_from_content_items(system.content_items)

SYSTEM_CONFIGS = [
    ("Vector-RAG",    "vector_only",  None),
    ("KRAG (a=1.0)",  "krag",        1.0),
    ("KRAG (a=0.5)",  "krag",        0.5),
    ("KRAG (a=0.3)",  "krag",        0.3),
]

EMOTION_ORDER = ["happiness", "sadness", "anger", "fear", "surprise", "disgust"]

def to_vec(target_emotions):
    if isinstance(target_emotions, dict):
        return np.array([target_emotions.get(e, 0.0) for e in EMOTION_ORDER])
    return np.asarray(target_emotions)

def to_sliders(target_emotions):
    vec = to_vec(target_emotions)
    return {e: int(round(v * 10)) for e, v in zip(EMOTION_ORDER, vec.tolist())}

print("Retrievers ready.")

---
# Part 1: Causal Necessity Analysis (FNS)

Perturbation-based analysis inspired by KG-SMILE (arXiv:2509.03626).

For each query, remove the dominant emotion edge from the KG,
invalidate the subgraph embedding cache, re-retrieve, and measure
the score drop.

$$\text{FNS} = \max(0,\; S_{\text{orig}} - S_{\text{perturbed}})$$

KRAG's combined score has range $[-(1-\alpha),\; \alpha]$ with max span of 1.0,
so the absolute drop is directly interpretable as a $[0, 1]$ necessity score.

Expected: BM25 / Vector-RAG FNS = 0 (no graph component).
KRAG FNS > 0, increasing as $\alpha$ decreases.

In [ ]:
def compute_fns_for_query(
    retriever_fn,
    knowledge_graph,
    subgraph_retriever,
    query_context,
    target_content_id,
    emotion_to_perturb,
    k=50,
):
    results_orig = retriever_fn(query_context, k=k)
    S_orig = 0.0
    for r in results_orig:
        if r.content_id == target_content_id:
            S_orig = r.combined_score
            break

    if S_orig <= 0:
        return 0.0

    edge_src = target_content_id
    edge_tgt = f"emotion_{emotion_to_perturb}"

    if not knowledge_graph.graph.has_edge(edge_src, edge_tgt):
        return 0.0

    edge_data_raw = knowledge_graph.graph.get_edge_data(edge_src, edge_tgt)
    if edge_data_raw is None:
        return 0.0
    edge_data = dict(edge_data_raw)
    if edge_data and isinstance(list(edge_data.values())[0], dict):
        edge_data = dict(list(edge_data.values())[0])

    knowledge_graph.graph.remove_edge(edge_src, edge_tgt)

    cached_emb = subgraph_retriever.subgraph_embeddings.pop(target_content_id, None)
    subgraph_retriever.index_subgraphs([target_content_id], hops=2)

    results_pert = retriever_fn(query_context, k=k)
    S_pert = 0.0
    for r in results_pert:
        if r.content_id == target_content_id:
            S_pert = r.combined_score
            break

    knowledge_graph.graph.add_edge(edge_src, edge_tgt, **edge_data)
    subgraph_retriever.subgraph_embeddings.pop(target_content_id, None)
    if cached_emb is not None:
        subgraph_retriever.subgraph_embeddings[target_content_id] = cached_emb
    else:
        subgraph_retriever.index_subgraphs([target_content_id], hops=2)

    return float(max(0.0, S_orig - S_pert))


print("FNS helper defined.")

In [ ]:
import contextlib, io

FNS_K = 50


def eval_fns_system(test_cases, rtype, alpha):
    if alpha is not None:
        system.config.alpha = alpha
    system.set_retriever_type(rtype)

    retriever = system.retriever

    fns_scores = []
    for i, tc in enumerate(test_cases):
        if i % 50 == 0:
            print(f"  query {i}/{len(test_cases)}", flush=True)

        dominant = max(tc.target_emotions, key=tc.target_emotions.get)

        emotion_profile = EmotionProfile(
            **{e: tc.target_emotions.get(e, 0.0) for e in EMOTION_ORDER}
        )
        embedding = system.content_embedder.embed_content(tc.query_text)
        qc = QueryContext(
            query_text=tc.query_text,
            user_emotions=emotion_profile,
            query_embedding=embedding,
            emotion_embedding=embedding,
        )

        results = retriever.retrieve(qc, k=FNS_K)
        if not results:
            continue
        target_id = results[0].content_id

        with contextlib.redirect_stdout(io.StringIO()):
            fns = compute_fns_for_query(
                retriever.retrieve,
                system.knowledge_graph,
                system.subgraph_retriever,
                qc, target_id, dominant, k=FNS_K,
            )
        fns_scores.append(fns)

    return fns_scores


print(f"FNS evaluation function defined (k={FNS_K}).")

### FNS — Agreement set

In [ ]:
system.response_generator.generate_response = lambda *a, **kw: ""

system.set_retriever_type("krag")
assert len(system.retriever.smoothed_emotions) == 0, (
    "smoothed_emotions must be empty for FNS -- perturbation requires live graph reads"
)

print(f"=== FNS — Agreement (n={len(agreement)}) ===\n")

fns_agr = {}

fns_agr["BM25"] = [0.0] * len(agreement)
print(f"BM25:       mean FNS = 0.0000  n={len(agreement)}  (no graph component)")

fns_agr["Vector-RAG"] = [0.0] * len(agreement)
print(f"Vector-RAG: mean FNS = 0.0000  n={len(agreement)}  (no graph component)")

In [ ]:
print("KRAG (a=1.0) — Agreement ...", flush=True)
fns_agr["KRAG (a=1.0)"] = eval_fns_system(agreement, "krag", 1.0)
print(f"  mean FNS = {np.mean(fns_agr['KRAG (a=1.0)']):.4f}  n={len(fns_agr['KRAG (a=1.0)'])}")

In [ ]:
print("KRAG (a=0.5) — Agreement ...", flush=True)
fns_agr["KRAG (a=0.5)"] = eval_fns_system(agreement, "krag", 0.5)
print(f"  mean FNS = {np.mean(fns_agr['KRAG (a=0.5)']):.4f}  n={len(fns_agr['KRAG (a=0.5)'])}")

In [ ]:
print("KRAG (a=0.3) — Agreement ...", flush=True)
fns_agr["KRAG (a=0.3)"] = eval_fns_system(agreement, "krag", 0.3)
print(f"  mean FNS = {np.mean(fns_agr['KRAG (a=0.3)']):.4f}  n={len(fns_agr['KRAG (a=0.3)'])}")

In [ ]:
system.response_generator.generate_response = lambda *a, **kw: ""

print(f"=== FNS — Dissonance (n={len(dissonance)}) ===\n")

fns_dis = {}

fns_dis["BM25"] = [0.0] * len(dissonance)
print(f"BM25:       mean FNS = 0.0000  n={len(dissonance)}  (no graph component)")

fns_dis["Vector-RAG"] = [0.0] * len(dissonance)
print(f"Vector-RAG: mean FNS = 0.0000  n={len(dissonance)}  (no graph component)")

In [ ]:
print("KRAG (a=1.0) — Dissonance ...", flush=True)
fns_dis["KRAG (a=1.0)"] = eval_fns_system(dissonance, "krag", 1.0)
print(f"  mean FNS = {np.mean(fns_dis['KRAG (a=1.0)']):.4f}  n={len(fns_dis['KRAG (a=1.0)'])}")

In [ ]:
print("KRAG (a=0.5) — Dissonance ...", flush=True)
fns_dis["KRAG (a=0.5)"] = eval_fns_system(dissonance, "krag", 0.5)
print(f"  mean FNS = {np.mean(fns_dis['KRAG (a=0.5)']):.4f}  n={len(fns_dis['KRAG (a=0.5)'])}")

In [ ]:
print("KRAG (a=0.3) — Dissonance ...", flush=True)
fns_dis["KRAG (a=0.3)"] = eval_fns_system(dissonance, "krag", 0.3)
print(f"  mean FNS = {np.mean(fns_dis['KRAG (a=0.3)']):.4f}  n={len(fns_dis['KRAG (a=0.3)'])}")

### FNS — Statistical tests

Paired comparisons: KRAG(a=0.3) vs each baseline.

In [ ]:
def paired_stats(a_scores, b_scores, label_a, label_b, metric):
    a = np.array(a_scores)
    b = np.array(b_scores)
    n = min(len(a), len(b))
    a, b = a[:n], b[:n]
    diff = a - b
    mean_d  = float(np.mean(diff))
    sd_diff = float(np.std(diff, ddof=1))
    dz      = mean_d / sd_diff if sd_diff > 0 else 0.0
    se      = sd_diff / np.sqrt(n)
    t_crit  = sp_stats.t.ppf(0.975, df=n - 1)
    ci      = (mean_d - t_crit * se, mean_d + t_crit * se)
    print(f"  {label_a} vs {label_b}: diff={mean_d:+.4f}  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]  dz={dz:+.3f}  n={n}")
    return {"mean_diff": mean_d, "ci": list(ci), "dz": dz, "n": n}


fns_stats = {}

COMPARISONS = ["BM25", "Vector-RAG", "KRAG (a=1.0)"]

print("\nFNS — Agreement paired tests (KRAG a=0.3 minus baseline)")
fns_stats["agreement"] = {}
for baseline in COMPARISONS:
    fns_stats["agreement"][baseline] = paired_stats(
        fns_agr["KRAG (a=0.3)"], fns_agr[baseline],
        "KRAG(a=0.3)", baseline, "FNS",
    )

print("\nFNS — Dissonance paired tests (KRAG a=0.3 minus baseline)")
fns_stats["dissonance"] = {}
for baseline in COMPARISONS:
    fns_stats["dissonance"][baseline] = paired_stats(
        fns_dis["KRAG (a=0.3)"], fns_dis[baseline],
        "KRAG(a=0.3)", baseline, "FNS",
    )

In [ ]:
import matplotlib.pyplot as plt

retrievers = list(fns_agr.keys())
x = np.arange(len(retrievers))
width = 0.35

agr_means = [np.mean(fns_agr[r]) for r in retrievers]
dis_means = [np.mean(fns_dis[r]) for r in retrievers]
agr_stds  = [np.std(fns_agr[r]) for r in retrievers]
dis_stds  = [np.std(fns_dis[r]) for r in retrievers]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, agr_means, width, yerr=agr_stds, label="Agreement", capsize=5)
ax.bar(x + width/2, dis_means, width, yerr=dis_stds, label="Dissonance", capsize=5)
ax.set_xticks(x)
ax.set_xticklabels(retrievers, fontsize=9, rotation=15)
ax.set_ylabel("FNS")
ax.set_title("Faithfulness Necessity Score by Retriever")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("/tmp/fns_bar.png", dpi=200, bbox_inches="tight")
plt.show()

---
# Part 2: RAGAS-Inspired Faithfulness (LLM Judge)

Adapted from RAGAS faithfulness (Es et al., arXiv:2309.15217) with
Gemini 3 Flash Preview as NLI judge, following LLM-as-Judge best
practices (arXiv:2411.15594).

**Adaptations from standard RAGAS:**
- Regex-based claim extraction (structured explanations, not free-form text)
- Verification against independently constructed KG evidence (not retrieved passages)
- Contextual claims only (numeric template consistency reported separately)

**Faithfulness = verified contextual claims / total contextual claims**

In [ ]:
from google import genai
from google.genai import types as genai_types


class VertexAIJudge:

    def __init__(self, project_id, model_name="gemini-2.5-flash", location="global"):
        self.client = genai.Client(
            vertexai=True,
            project=project_id,
            location=location,
            http_options=genai_types.HttpOptions(
                retry_options=genai_types.HttpRetryOptions(
                    initial_delay=1.0,
                    attempts=5,
                    exp_base=2,
                    max_delay=60,
                    jitter=1,
                    http_status_codes=[408, 429, 500, 502, 503, 504],
                ),
                timeout=120 * 1000,
            ),
        )
        self.model_name = model_name
        self.config = genai_types.GenerateContentConfig(temperature=0)

    def generate(self, prompt):
        response = self.client.models.generate_content(
            model=self.model_name, contents=prompt, config=self.config,
        )
        return response.text


llm_judge = VertexAIJudge(project_id=PROJECT_ID)
test_response = llm_judge.generate("Say 'OK' if you can read this.")
print(f"LLM judge ready: {llm_judge.model_name} -> {test_response.strip()}")
print("Retry: 5 attempts, exponential backoff 1-60s, jitter enabled")

In [ ]:
import re


def verify_numeric_claims(explanation, result):
    verdicts = []
    tol = 0.005

    sem_match = re.search(
        r'(?:Semantic similarity[^:]*:\s*|(?:High|Moderate) semantic \()(\d+\.\d+)',
        explanation,
    )
    if sem_match:
        claimed = float(sem_match.group(1))
        actual = result.semantic_score
        verdicts.append((
            f"Semantic score = {claimed:.3f} (actual: {actual:.3f})",
            abs(claimed - actual) < tol,
        ))

    graph_match = re.search(
        r'(?:Graph similarity[^:]*:\s*|(?:Strong|Moderate) graph \()(\d+\.\d+)',
        explanation,
    )
    if graph_match:
        claimed = float(graph_match.group(1))
        actual = result.knowledge_score
        verdicts.append((
            f"Graph score = {claimed:.3f} (actual: {actual:.3f})",
            abs(claimed - actual) < tol,
        ))

    combined_match = re.search(r'=\s*(\d+\.\d+)\]?\s*$', explanation)
    if combined_match:
        claimed = float(combined_match.group(1))
        actual = result.combined_score
        verdicts.append((
            f"Combined score = {claimed:.3f} (actual: {actual:.3f})",
            abs(claimed - actual) < tol,
        ))

    return verdicts


print("Tier 1 numeric verification defined.")

In [ ]:
ragas_evaluator = RAGASFaithfulnessEvaluator(
    llm_client=llm_judge,
    model_name=llm_judge.model_name,
)


def evaluate_single_explanation(retriever, query_context, result, knowledge_graph):
    explanation = retriever.explain_retrieval(result, query_context)
    graph_context = build_full_subgraph_context(
        result.content_id, knowledge_graph, hops=2
    )

    tier1_verdicts = verify_numeric_claims(explanation, result)
    tier1_verified = sum(1 for _, v in tier1_verdicts if v)
    tier1_total = len(tier1_verdicts)

    contextual_claims = ragas_evaluator.extract_claims(explanation)
    contextual_claims = [c for c in contextual_claims if c.claim_type == "contextual"]

    tier2_verdicts = []
    tier2_verified = 0
    for claim in contextual_claims:
        is_supported = ragas_evaluator._verify_claim(claim.text, graph_context)
        tier2_verdicts.append({
            "claim": claim.text,
            "verdict": "SUPPORTED" if is_supported else "NOT_SUPPORTED",
        })
        if is_supported:
            tier2_verified += 1
    tier2_total = len(contextual_claims)

    faithfulness = tier2_verified / tier2_total if tier2_total > 0 else 1.0

    return {
        "faithfulness": faithfulness,
        "template_consistency": tier1_verified / tier1_total if tier1_total > 0 else 1.0,
        "tier1_verified": tier1_verified,
        "tier1_total": tier1_total,
        "tier2_verified": tier2_verified,
        "tier2_total": tier2_total,
        "explanation": explanation,
        "graph_context": graph_context,
        "tier1_verdicts": tier1_verdicts,
        "tier2_verdicts": tier2_verdicts,
    }


print("Faithfulness evaluation pipeline defined.")

### LLM judge calibration

Inject synthetic unfaithful claims to measure the judge's false-positive rate.
If the judge correctly rejects fabricated claims, we can trust its verdicts.

In [ ]:
import random

sample_ids = random.sample(content_ids, min(10, len(content_ids)))
FAKE_GENRES = ["Western", "Musical", "Documentary", "Film-Noir", "IMAX"]

true_claims = []
fake_claims = []

for cid in sample_ids:
    ctx = build_full_subgraph_context(cid, system.knowledge_graph, hops=2)
    if not ctx:
        continue
    genres = []
    for line in ctx.split("\n"):
        if "GENRES IN SUBGRAPH" in line:
            genres = [g.strip() for g in line.split(":")[1].split(",")]
            break
    if genres:
        true_claim = f"Graph context states: belongs to genre {genres[0]}"
        true_claims.append((true_claim, ctx))

        available_fakes = [g for g in FAKE_GENRES if g.lower() not in ctx.lower()]
        if available_fakes:
            fake_claim = f"Graph context states: belongs to genre {random.choice(available_fakes)}"
            fake_claims.append((fake_claim, ctx))

tp, fn, tn, fp = 0, 0, 0, 0

print("Calibrating LLM judge with true claims ...")
for claim, ctx in true_claims:
    if ragas_evaluator._verify_claim(claim, ctx):
        tp += 1
    else:
        fn += 1

print("Calibrating LLM judge with fabricated claims ...")
for claim, ctx in fake_claims:
    if ragas_evaluator._verify_claim(claim, ctx):
        fp += 1
    else:
        tn += 1

total_true = tp + fn
total_fake = tn + fp

print(f"\n=== LLM Judge Calibration ===")
if total_true > 0:
    print(f"True claims:  {tp}/{total_true} correct  (FNR = {fn/total_true:.2f})")
if total_fake > 0:
    print(f"Fake claims:  {tn}/{total_fake} rejected (FPR = {fp/total_fake:.2f})")

### Run faithfulness evaluation

Sample 200 agreement + 200 dissonance queries, top-3 results per query,
for each KRAG variant (a=1.0, 0.5, 0.3). ~1,200 evaluations per variant.

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

np.random.seed(42)
agr_sample_idx = np.random.choice(len(agreement), size=min(200, len(agreement)), replace=False)
dis_sample_idx = np.random.choice(len(dissonance), size=min(200, len(dissonance)), replace=False)
faith_queries = (
    [(agreement[i], "agreement") for i in agr_sample_idx] +
    [(dissonance[i], "dissonance") for i in dis_sample_idx]
)
print(f"Faithfulness sample: {len(faith_queries)} queries")

faith_results = {}
faith_per_query = {}
all_verdicts_for_meta = []

FAITH_WORKERS = 3
_api_semaphore = threading.Semaphore(FAITH_WORKERS)


def _eval_one(args):
    retriever, qc, result, kg = args
    with _api_semaphore:
        return evaluate_single_explanation(retriever, qc, result, kg)


def run_faithfulness_variant(name, rtype, alpha):
    system.config.alpha = alpha
    system.set_retriever_type(rtype)
    retriever = system.retriever

    tasks = []
    for qi, (tc, qtype) in enumerate(faith_queries):
        emotion_profile = EmotionProfile(
            **{e: tc.target_emotions.get(e, 0.0) for e in EMOTION_ORDER}
        )
        embedding = system.content_embedder.embed_content(tc.query_text)
        qc = QueryContext(
            query_text=tc.query_text,
            user_emotions=emotion_profile,
            query_embedding=embedding,
            emotion_embedding=embedding,
        )
        results = retriever.retrieve(qc, k=3)
        for result in results:
            tasks.append((retriever, qc, result, system.knowledge_graph))

    print(f"  {len(tasks)} evaluations, {FAITH_WORKERS} concurrent workers", flush=True)

    scores = []
    template_checks = []
    errors = 0
    done = 0

    with ThreadPoolExecutor(max_workers=FAITH_WORKERS) as pool:
        futures = {pool.submit(_eval_one, t): t for t in tasks}
        for future in as_completed(futures):
            try:
                eval_out = future.result()
            except Exception as e:
                errors += 1
                if errors <= 3:
                    print(f"  [warn] eval failed: {e}", flush=True)
                continue

            scores.append(eval_out["faithfulness"])
            template_checks.append(eval_out["template_consistency"])

            for v in eval_out["tier2_verdicts"]:
                all_verdicts_for_meta.append({
                    "variant": name,
                    "claim": v["claim"],
                    "verdict": v["verdict"],
                    "graph_context": eval_out["graph_context"][:500],
                })

            done += 1
            if done % 100 == 0:
                print(f"  {done}/{len(tasks)} done ({errors} errors)", flush=True)

    faith_results[name] = {
        "mean_faithfulness": float(np.mean(scores)),
        "std_faithfulness": float(np.std(scores)),
        "template_consistency": float(np.mean(template_checks)),
        "perfect_ratio": float(sum(1 for s in scores if s >= 0.999) / len(scores)),
        "n": len(scores),
        "errors": errors,
    }
    faith_per_query[name] = scores

    print(f"  Faithfulness (contextual): {faith_results[name]['mean_faithfulness']:.4f}")
    print(f"  Template consistency:      {faith_results[name]['template_consistency']:.4f}")
    print(f"  Perfect ratio:             {faith_results[name]['perfect_ratio']:.4f}")
    print(f"  Completed: {len(scores)}, Errors: {errors}")


print(f"Faithfulness helper defined ({FAITH_WORKERS} concurrent, SDK retry enabled).")

In [ ]:
print("KRAG (a=1.0) — Faithfulness ...", flush=True)
run_faithfulness_variant("KRAG (a=1.0)", "krag", 1.0)

In [ ]:
print("KRAG (a=0.5) — Faithfulness ...", flush=True)
run_faithfulness_variant("KRAG (a=0.5)", "krag", 0.5)

In [ ]:
print("KRAG (a=0.3) — Faithfulness ...", flush=True)
run_faithfulness_variant("KRAG (a=0.3)", "krag", 0.3)

### Faithfulness statistical tests

In [ ]:
faith_stats = {}

print("Faithfulness paired test: KRAG(a=0.3) vs KRAG(a=1.0)")
faith_stats["a03_vs_a10"] = paired_stats(
    faith_per_query["KRAG (a=0.3)"],
    faith_per_query["KRAG (a=1.0)"],
    "KRAG(a=0.3)", "KRAG(a=1.0)", "Faith",
)

print("\nFaithfulness paired test: KRAG(a=0.5) vs KRAG(a=1.0)")
faith_stats["a05_vs_a10"] = paired_stats(
    faith_per_query["KRAG (a=0.5)"],
    faith_per_query["KRAG (a=1.0)"],
    "KRAG(a=0.5)", "KRAG(a=1.0)", "Faith",
)

In [ ]:
krag_names = ["KRAG (a=1.0)", "KRAG (a=0.5)", "KRAG (a=0.3)"]
x = np.arange(len(krag_names))

faith_means = [faith_results[n]["mean_faithfulness"] for n in krag_names]
faith_stds  = [faith_results[n]["std_faithfulness"] for n in krag_names]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x, faith_means, 0.5, yerr=faith_stds, capsize=5, color="steelblue")
ax.set_xticks(x)
ax.set_xticklabels(krag_names, fontsize=10)
ax.set_ylabel("Contextual Faithfulness")
ax.set_title("RAGAS-Inspired Faithfulness by KRAG Variant")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("/tmp/faithfulness_bar.png", dpi=200, bbox_inches="tight")
plt.show()

---
## Meta-evaluation

Export 100 stratified (claim, context, LLM verdict) triples for manual annotation.
Stratified by verdict: 50 SUPPORTED + 50 NOT_SUPPORTED to ensure
meaningful false-positive/false-negative coverage.

In [ ]:
np.random.seed(123)

supported = [v for v in all_verdicts_for_meta if v["verdict"] == "SUPPORTED"]
not_supported = [v for v in all_verdicts_for_meta if v["verdict"] == "NOT_SUPPORTED"]

n_per_stratum = 50
sup_sample = [supported[i] for i in np.random.choice(
    len(supported), size=min(n_per_stratum, len(supported)), replace=False
)]
nsup_sample = [not_supported[i] for i in np.random.choice(
    len(not_supported), size=min(n_per_stratum, len(not_supported)), replace=False
)]

meta_samples = sup_sample + nsup_sample
np.random.shuffle(meta_samples)

print(f"Exported {len(meta_samples)} samples for manual annotation.")
print(f"  SUPPORTED:     {len(sup_sample)}")
print(f"  NOT_SUPPORTED: {len(nsup_sample)}")
print(f"  Total pool:    {len(all_verdicts_for_meta)} "
      f"({len(supported)} sup, {len(not_supported)} not_sup)")

with open("/tmp/meta_evaluation_samples.json", "w") as f:
    json.dump(meta_samples, f, indent=2)

---
## Save results

### Part 1: FNS

In [ ]:
def summarize_fns(fns_dict):
    return {
        name: {
            "mean": float(np.mean(scores)),
            "std": float(np.std(scores)),
            "median": float(np.median(scores)),
            "n": len(scores),
        }
        for name, scores in fns_dict.items()
    }


s_orig_diagnostic = {}
for alpha in [1.0, 0.5, 0.3]:
    system.config.alpha = alpha
    system.set_retriever_type("krag")
    retriever = system.retriever

    s_values = []
    for tc in agreement:
        emotion_profile = EmotionProfile(
            **{e: tc.target_emotions.get(e, 0.0) for e in EMOTION_ORDER}
        )
        embedding = system.content_embedder.embed_content(tc.query_text)
        qc = QueryContext(
            query_text=tc.query_text,
            user_emotions=emotion_profile,
            query_embedding=embedding,
            emotion_embedding=embedding,
        )
        results = retriever.retrieve(qc, k=FNS_K)
        if results:
            s_values.append(results[0].combined_score)

    vals = np.array(s_values)
    n_neg = int(np.sum(vals <= 0))
    s_orig_diagnostic[f"a={alpha}"] = {
        "mean_s_orig": float(vals.mean()),
        "median_s_orig": float(np.median(vals)),
        "n_nonpositive": n_neg,
        "pct_nonpositive": float(n_neg / len(vals) * 100),
        "n": len(vals),
    }
    print(f"a={alpha}: S_orig<=0: {n_neg}/{len(vals)} ({n_neg/len(vals)*100:.1f}%)  mean={vals.mean():.4f}")


part1_out = {
    "metadata": {
        "experiment": "explainability_evaluation_part1_fns",
        "encoder": "krag_encoder_aw0.3",
        "n_agreement": len(agreement),
        "n_dissonance": len(dissonance),
        "device": DEVICE,
    },
    "fns": {
        "agreement": summarize_fns(fns_agr),
        "dissonance": summarize_fns(fns_dis),
    },
    "fns_floor_effect": s_orig_diagnostic,
    "statistical_tests": {
        "fns_agreement": fns_stats.get("agreement", {}),
        "fns_dissonance": fns_stats.get("dissonance", {}),
    },
    "per_query": {
        "fns_agreement": {k: v for k, v in fns_agr.items()},
        "fns_dissonance": {k: v for k, v in fns_dis.items()},
    },
}

with open("/tmp/explainability_part1_fns.json", "w") as f:
    json.dump(part1_out, f, indent=2)

from google.colab.files import download
download("/tmp/explainability_part1_fns.json")
download("/tmp/fns_bar.png")
print("Part 1 (FNS) results saved.")

### Part 2: Faithfulness + combined

In [ ]:
part1_path = "/tmp/explainability_part1_fns.json"
with open(part1_path) as f:
    part1_data = json.load(f)

part2_out = {
    "metadata": {
        **part1_data["metadata"],
        "experiment": "explainability_evaluation_complete",
        "judge_model": llm_judge.model_name,
        "faith_sample_size": len(faith_queries),
        "faith_top_k": 3,
        "faith_workers": FAITH_WORKERS,
    },
    "fns": part1_data["fns"],
    "fns_floor_effect": part1_data["fns_floor_effect"],
    "faithfulness": faith_results,
    "statistical_tests": {
        **part1_data["statistical_tests"],
        "faithfulness": faith_stats,
    },
    "per_query": {
        **part1_data["per_query"],
        "faithfulness": {k: v for k, v in faith_per_query.items()},
    },
    "meta_evaluation": {
        "n_samples": len(meta_samples),
        "n_supported": len(sup_sample),
        "n_not_supported": len(nsup_sample),
        "total_verdicts_pool": len(all_verdicts_for_meta),
    },
}

with open("/tmp/explainability_results.json", "w") as f:
    json.dump(part2_out, f, indent=2)

from google.colab.files import download
download("/tmp/explainability_results.json")
download("/tmp/faithfulness_bar.png")
download("/tmp/meta_evaluation_samples.json")
print("Part 2 (Faithfulness + combined) results saved.")

---
# Part 3: Full-Profile Ablation FNS

The single-edge perturbation (Part 1) removes only the dominant emotion edge.
Here we ablate the **entire emotional profile** — all 6 emotion edges for
the target movie — giving a stronger perturbation that tests whether the
movie's KG emotion profile is causally necessary for its ranking.

We also report **conditional FNS** (FNS | S_orig > 0) alongside overall FNS
to separate the causal effect from the floor-effect dilution at low alpha.

In [ ]:
def compute_fns_full_ablation(
    retriever_fn,
    knowledge_graph,
    subgraph_retriever,
    query_context,
    target_content_id,
    k=50,
):
    results_orig = retriever_fn(query_context, k=k)
    S_orig = 0.0
    for r in results_orig:
        if r.content_id == target_content_id:
            S_orig = r.combined_score
            break

    if S_orig <= 0:
        return 0.0, S_orig

    removed_edges = []
    for emotion in EMOTION_ORDER:
        edge_tgt = f"emotion_{emotion}"
        if not knowledge_graph.graph.has_edge(target_content_id, edge_tgt):
            continue
        edge_data_raw = knowledge_graph.graph.get_edge_data(target_content_id, edge_tgt)
        if edge_data_raw is None:
            continue
        edge_data = dict(edge_data_raw)
        if edge_data and isinstance(list(edge_data.values())[0], dict):
            edge_data = dict(list(edge_data.values())[0])
        knowledge_graph.graph.remove_edge(target_content_id, edge_tgt)
        removed_edges.append((target_content_id, edge_tgt, edge_data))

    if not removed_edges:
        return 0.0, S_orig

    cached_emb = subgraph_retriever.subgraph_embeddings.pop(target_content_id, None)
    subgraph_retriever.index_subgraphs([target_content_id], hops=2)

    results_pert = retriever_fn(query_context, k=k)
    S_pert = 0.0
    for r in results_pert:
        if r.content_id == target_content_id:
            S_pert = r.combined_score
            break

    for src, tgt, data in removed_edges:
        knowledge_graph.graph.add_edge(src, tgt, **data)
    subgraph_retriever.subgraph_embeddings.pop(target_content_id, None)
    if cached_emb is not None:
        subgraph_retriever.subgraph_embeddings[target_content_id] = cached_emb
    else:
        subgraph_retriever.index_subgraphs([target_content_id], hops=2)

    return float(max(0.0, S_orig - S_pert)), S_orig


def eval_fns_full(test_cases, rtype, alpha):
    system.config.alpha = alpha
    system.set_retriever_type(rtype)
    retriever = system.retriever

    fns_scores = []
    s_orig_values = []
    for i, tc in enumerate(test_cases):
        if i % 50 == 0:
            print(f"  query {i}/{len(test_cases)}", flush=True)

        emotion_profile = EmotionProfile(
            **{e: tc.target_emotions.get(e, 0.0) for e in EMOTION_ORDER}
        )
        embedding = system.content_embedder.embed_content(tc.query_text)
        qc = QueryContext(
            query_text=tc.query_text,
            user_emotions=emotion_profile,
            query_embedding=embedding,
            emotion_embedding=embedding,
        )

        results = retriever.retrieve(qc, k=FNS_K)
        if not results:
            continue
        target_id = results[0].content_id

        with contextlib.redirect_stdout(io.StringIO()):
            fns, s_orig = compute_fns_full_ablation(
                retriever.retrieve,
                system.knowledge_graph,
                system.subgraph_retriever,
                qc, target_id, k=FNS_K,
            )
        fns_scores.append(fns)
        s_orig_values.append(s_orig)

    return fns_scores, s_orig_values


print("Full-profile ablation FNS helpers defined.")

### Full-profile FNS — Agreement

In [ ]:
system.response_generator.generate_response = lambda *a, **kw: ""

print(f"=== Full-Profile FNS — Agreement (n={len(agreement)}) ===\n")

fp_fns_agr = {}
fp_sorig_agr = {}

fp_fns_agr["BM25"] = [0.0] * len(agreement)
fp_sorig_agr["BM25"] = [0.0] * len(agreement)
print(f"BM25:       mean FNS = 0.0000  (no graph)")

fp_fns_agr["Vector-RAG"] = [0.0] * len(agreement)
fp_sorig_agr["Vector-RAG"] = [0.0] * len(agreement)
print(f"Vector-RAG: mean FNS = 0.0000  (no graph)")

In [ ]:
print("KRAG (a=1.0) — Full-profile Agreement ...", flush=True)
fp_fns_agr["KRAG (a=1.0)"], fp_sorig_agr["KRAG (a=1.0)"] = eval_fns_full(agreement, "krag", 1.0)
scores = fp_fns_agr["KRAG (a=1.0)"]
print(f"  mean FNS = {np.mean(scores):.4f}  n={len(scores)}")

In [ ]:
print("KRAG (a=0.5) — Full-profile Agreement ...", flush=True)
fp_fns_agr["KRAG (a=0.5)"], fp_sorig_agr["KRAG (a=0.5)"] = eval_fns_full(agreement, "krag", 0.5)
scores = fp_fns_agr["KRAG (a=0.5)"]
print(f"  mean FNS = {np.mean(scores):.4f}  n={len(scores)}")

In [ ]:
print("KRAG (a=0.3) — Full-profile Agreement ...", flush=True)
fp_fns_agr["KRAG (a=0.3)"], fp_sorig_agr["KRAG (a=0.3)"] = eval_fns_full(agreement, "krag", 0.3)
scores = fp_fns_agr["KRAG (a=0.3)"]
print(f"  mean FNS = {np.mean(scores):.4f}  n={len(scores)}")

### Full-profile FNS — Dissonance

In [ ]:
print(f"=== Full-Profile FNS — Dissonance (n={len(dissonance)}) ===\n")

fp_fns_dis = {}
fp_sorig_dis = {}

fp_fns_dis["BM25"] = [0.0] * len(dissonance)
fp_sorig_dis["BM25"] = [0.0] * len(dissonance)
print(f"BM25:       mean FNS = 0.0000  (no graph)")

fp_fns_dis["Vector-RAG"] = [0.0] * len(dissonance)
fp_sorig_dis["Vector-RAG"] = [0.0] * len(dissonance)
print(f"Vector-RAG: mean FNS = 0.0000  (no graph)")

In [ ]:
print("KRAG (a=1.0) — Full-profile Dissonance ...", flush=True)
fp_fns_dis["KRAG (a=1.0)"], fp_sorig_dis["KRAG (a=1.0)"] = eval_fns_full(dissonance, "krag", 1.0)
scores = fp_fns_dis["KRAG (a=1.0)"]
print(f"  mean FNS = {np.mean(scores):.4f}  n={len(scores)}")

In [ ]:
print("KRAG (a=0.5) — Full-profile Dissonance ...", flush=True)
fp_fns_dis["KRAG (a=0.5)"], fp_sorig_dis["KRAG (a=0.5)"] = eval_fns_full(dissonance, "krag", 0.5)
scores = fp_fns_dis["KRAG (a=0.5)"]
print(f"  mean FNS = {np.mean(scores):.4f}  n={len(scores)}")

In [ ]:
print("KRAG (a=0.3) — Full-profile Dissonance ...", flush=True)
fp_fns_dis["KRAG (a=0.3)"], fp_sorig_dis["KRAG (a=0.3)"] = eval_fns_full(dissonance, "krag", 0.3)
scores = fp_fns_dis["KRAG (a=0.3)"]
print(f"  mean FNS = {np.mean(scores):.4f}  n={len(scores)}")

### Full-profile FNS — Results summary and conditional FNS

In [ ]:
print("=" * 70)
print("FULL-PROFILE ABLATION FNS — SUMMARY")
print("=" * 70)

krag_variants = ["KRAG (a=1.0)", "KRAG (a=0.5)", "KRAG (a=0.3)"]

fp_conditional = {}

for condition, fns_dict, sorig_dict in [
    ("Agreement", fp_fns_agr, fp_sorig_agr),
    ("Dissonance", fp_fns_dis, fp_sorig_dis),
]:
    print(f"\n--- {condition} ---")
    print(f"{'Retriever':<18} {'Overall FNS':>12} {'Cond. FNS':>12} "
          f"{'S_orig>0':>10} {'n':>6}")
    print("-" * 62)

    for name in ["BM25", "Vector-RAG"] + krag_variants:
        scores = np.array(fns_dict[name])
        s_origs = np.array(sorig_dict[name])

        mask = s_origs > 0
        n_pos = int(mask.sum())
        cond_fns = float(np.mean(scores[mask])) if n_pos > 0 else 0.0

        fp_conditional[f"{name}_{condition}"] = {
            "overall_mean": float(np.mean(scores)),
            "overall_std": float(np.std(scores)),
            "conditional_mean": cond_fns,
            "conditional_std": float(np.std(scores[mask])) if n_pos > 0 else 0.0,
            "n_positive": n_pos,
            "n_total": len(scores),
            "pct_positive": float(n_pos / len(scores) * 100),
        }

        print(f"{name:<18} {np.mean(scores):>12.4f} {cond_fns:>12.4f} "
              f"{n_pos:>7}/{len(scores):<3} {len(scores):>6}")

print("\n\nComparison: Single-edge (Part 1) vs Full-profile (Part 3)")
print("-" * 62)
print(f"{'Variant':<18} {'Single-edge':>12} {'Full-prof.':>12} {'Ratio':>8}")
print("-" * 62)
for name in krag_variants:
    if name in fns_agr and name in fp_fns_agr:
        se = np.mean(fns_agr[name])
        fp = np.mean(fp_fns_agr[name])
        ratio = fp / se if se > 0 else float('inf')
        print(f"{name:<18} {se:>12.4f} {fp:>12.4f} {ratio:>8.2f}x")

In [ ]:
fp_stats = {}

print("Full-profile FNS — Agreement paired tests (KRAG a=0.3 minus baseline)")
fp_stats["agreement"] = {}
for baseline in ["BM25", "Vector-RAG", "KRAG (a=1.0)"]:
    fp_stats["agreement"][baseline] = paired_stats(
        fp_fns_agr["KRAG (a=0.3)"], fp_fns_agr[baseline],
        "KRAG(a=0.3)", baseline, "FNS-FP",
    )

print("\nFull-profile FNS — Dissonance paired tests (KRAG a=0.3 minus baseline)")
fp_stats["dissonance"] = {}
for baseline in ["BM25", "Vector-RAG", "KRAG (a=1.0)"]:
    fp_stats["dissonance"][baseline] = paired_stats(
        fp_fns_dis["KRAG (a=0.3)"], fp_fns_dis[baseline],
        "KRAG(a=0.3)", baseline, "FNS-FP",
    )

print("\nFull-profile FNS — Cross-alpha (a=0.5 vs a=0.3)")
fp_stats["a05_vs_a03_agr"] = paired_stats(
    fp_fns_agr["KRAG (a=0.5)"], fp_fns_agr["KRAG (a=0.3)"],
    "KRAG(a=0.5)", "KRAG(a=0.3)", "FNS-FP Agr",
)
fp_stats["a05_vs_a03_dis"] = paired_stats(
    fp_fns_dis["KRAG (a=0.5)"], fp_fns_dis["KRAG (a=0.3)"],
    "KRAG(a=0.5)", "KRAG(a=0.3)", "FNS-FP Dis",
)

In [ ]:
retrievers = list(fp_fns_agr.keys())
x = np.arange(len(retrievers))
width = 0.35

agr_means = [np.mean(fp_fns_agr[r]) for r in retrievers]
dis_means = [np.mean(fp_fns_dis[r]) for r in retrievers]
agr_stds  = [np.std(fp_fns_agr[r]) for r in retrievers]
dis_stds  = [np.std(fp_fns_dis[r]) for r in retrievers]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, agr_means, width, yerr=agr_stds, label="Agreement", capsize=5)
ax.bar(x + width/2, dis_means, width, yerr=dis_stds, label="Dissonance", capsize=5)
ax.set_xticks(x)
ax.set_xticklabels(retrievers, fontsize=9, rotation=15)
ax.set_ylabel("FNS (Full-Profile Ablation)")
ax.set_title("Full-Profile Ablation FNS by Retriever")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("/tmp/fns_full_profile_bar.png", dpi=200, bbox_inches="tight")
plt.show()

### Save Part 3 results

In [ ]:
def summarize_fp_fns(fns_dict, sorig_dict):
    out = {}
    for name, scores in fns_dict.items():
        s_arr = np.array(scores)
        so_arr = np.array(sorig_dict[name])
        mask = so_arr > 0
        n_pos = int(mask.sum())
        out[name] = {
            "overall_mean": float(np.mean(s_arr)),
            "overall_std": float(np.std(s_arr)),
            "overall_median": float(np.median(s_arr)),
            "conditional_mean": float(np.mean(s_arr[mask])) if n_pos > 0 else 0.0,
            "conditional_std": float(np.std(s_arr[mask])) if n_pos > 0 else 0.0,
            "n_positive_sorig": n_pos,
            "n": len(scores),
        }
    return out


part3_out = {
    "metadata": {
        "experiment": "explainability_part3_full_profile_fns",
        "encoder": "krag_encoder_aw0.3",
        "perturbation": "all_emotion_edges",
        "n_agreement": len(agreement),
        "n_dissonance": len(dissonance),
        "device": DEVICE,
    },
    "fns_full_profile": {
        "agreement": summarize_fp_fns(fp_fns_agr, fp_sorig_agr),
        "dissonance": summarize_fp_fns(fp_fns_dis, fp_sorig_dis),
    },
    "conditional_fns": fp_conditional,
    "statistical_tests": fp_stats,
    "per_query": {
        "fns_agreement": {k: v for k, v in fp_fns_agr.items()},
        "fns_dissonance": {k: v for k, v in fp_fns_dis.items()},
        "s_orig_agreement": {k: v for k, v in fp_sorig_agr.items()},
        "s_orig_dissonance": {k: v for k, v in fp_sorig_dis.items()},
    },
}

with open("/tmp/explainability_part3_full_profile_fns.json", "w") as f:
    json.dump(part3_out, f, indent=2)

from google.colab.files import download
download("/tmp/explainability_part3_full_profile_fns.json")
download("/tmp/fns_full_profile_bar.png")
print("Part 3 (Full-profile FNS) results saved.")